In [1]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plot
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
import sklearn.metrics
from sklearn.metrics import explained_variance_score, mean_squared_error, r2_score # https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html
import statsmodels.api as sm
import pandas
import matplotlib.pyplot as plt
import scipy as sp
from scipy.odr import *
import random
import operator
import os

# For 3d plots. This import is necessary to have 3D plotting below
from mpl_toolkits.mplot3d import Axes3D

# For statistics. Requires statsmodels 5.0 or more
from statsmodels.formula.api import ols
from statsmodels.formula.api import wls
# Analysis of Variance (ANOVA) on linear models
from statsmodels.stats.anova import anova_lm

# Adapted from:
# https://www.datasciencecentral.com/profiles/blogs/linear-regression-in-python-use-of-numpy-scipy-and-statsmodels,
# https://realpython.com/linear-regression-in-python/#multiple-linear-regression

In [2]:
def partition_check (source_array, split_arrays):
    index_list_source = []
    index_list_partitioned = []
    partitioned_array = np.concatenate(split_arrays)
    for i in range (0, len(source_array)):
        active_check = source_array[i]
        for k in range(0, len(partitioned_array)):
            if active_check == partitioned_array[k]:
                index_list_source.append(i)
                index_list_partitioned.append(k)
    if len(index_list_source) == len(index_list_partitioned):
        print('Checked: Number of sources match after partition.')
    else:
        print('Error: Number of sources does not match after partition.')
    order_check = 1
    for i in range (0, len(index_list_source)):
        if index_list_source[i] != index_list_partitioned[i]:
            order_check = 0
    if order_check == 1:
        print('Checked: All sources accounted for, in original order.')
    else:
        print ('Error: Sources do not appear in correct order after partition.')

In [3]:
simple_source = [1,2,3,4,5,6]
simple_test_array = [[1,2,3],[4,5,6]]
simple_test_array_2 = [[4,5,6],[1,2,3]]

In [4]:
partition_check(simple_source,simple_test_array)

Checked: Number of sources match after partition.
Checked: All sources accounted for, in original order.


In [5]:
def calc_rms (data_set):
    sum_of_squares = 0
    for i in range(0,len(data_set)):
        sum_of_squares = sum_of_squares + data_set[i]**2
    rms = np.sqrt(sum_of_squares/len(data_set))
    return rms

def calc_rms_error (predicted_data, obsv_data):
    sum_of_squares_of_error = 0
    if len(predicted_data) != len(obsv_data):
        print ('Warning! Data Sets of Unequal Size! Calculation is invalid!')
    for i in range(0,len(predicted_data)):
        sum_of_squares_of_error = sum_of_squares_of_error + (predicted_data[i] - obsv_data[i])**2
    ms_error = sum_of_squares_of_error/len(predicted_data)
    rms_error = np.sqrt(ms_error)
    return rms_error
    
lum_limits=(2.8,4.5)
per_limits=(1.2,3)
mass_limits=(np.log(0.5),np.log(5))
Q_limits=(-0.65,1.45)
mdot_limits=(-9,-4)
radius_limits=(50,650)
resid_limits=(-3,3)

In [6]:
M0StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FundamentalMStarData.csv',delimiter=',',skip_header=0))) # No header, so don't skip
M1StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FirstOvertoneMStarData.csv',delimiter=',',skip_header=1))) # Has header, so skip
C0StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FundamentalCStarData.csv',delimiter=',',skip_header=1))) # Has header, so skip
C1StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FirstOvertoneCStarData.csv',delimiter=',',skip_header=0))) # No header, so don't skip

In [7]:
MdotIndex=5 # Column of imported files that has Mdot data
MdotErrorIndex=6 # Absolute error in Mdot
#MdotIndex=2 # Column of imported files that has Mdot_dust Data
LIndex=8 # Column of imported files that has LBol data
LErrorIndex=9 # Absolute error in L
PIndex=13 # Column of imported files that has Period data
MIndex=22 # Column of imported files that has Mass data
RIndex=23 # Column of imported files that has Radius data
TeffObsvIndex=12 # Column of imported files that has Observed Teff Data
TeffCalcIndex=24 # Column of imported files that has Calculated Teff Data
MdotperMIndex_sorting=7

# For Qx Method
MdotIndex_sorting=4
LIndex_sorting=2
PIndex_sorting=3
MIndex_sorting=0
RIndex_sorting=1
TeffObsvIndex_sorting=5
TeffCalcIndex_sorting=6
MdotperMIndex_sorting=7


In [8]:
# Functions to sort a list
def examine_index_Q(elem): 
    # Sorting Key. Will sort a list based on the specified index hard coded here.
    return elem[MdotIndex]

def sort_star_data_Q(DataToSort):
    TransposedArray=np.transpose(DataToSort)
    TempList=TransposedArray.tolist()
    TempList=sorted(TempList,key = operator.itemgetter(MdotIndex))
    return np.transpose(np.asarray(TempList))

def MassRangeSelect_Q (DataToTrim, LowerBound, UpperBound ):
    return np.transpose(np.transpose(sort_star_data_Q(DataToTrim))[LowerBound-1:UpperBound-1])

In [9]:
def VWFit_Q_method (data_set, specify_star_type ='', VW_fit_lower_percentile_bound=0, VW_fit_upper_percentile_bound = 100 , save_figures_VW = 0, method_folder = 'Percentile_Range'): 
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/C1/'
    else:
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/Other/'
    
    # First, set up data for the initial VW Fit
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    PercentileLower= int(np.floor(VW_fit_lower_percentile_bound/100*len(np.transpose(DataToFit))))
    PercentileUpper = int(np.ceil(VW_fit_upper_percentile_bound/100*len(np.transpose(DataToFit))))
    #print(VW_fit_lower_percentile_bound, 'th Percentile:', PercentileLower,VW_fit_upper_percentile_bound, 'th Percentile:', PercentileUpper)
    
    if PercentileLower !=0 and PercentileUpper != 100:
        DataToFit = MassRangeSelect_Q (DataToFit, PercentileLower, PercentileUpper )
    
    #print('DEBUG: DataToFit:', DataToFit)
    
    
    # Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    
    # Next, do the fit to obtain V and W
    ## logP ~ V + W log L
    ## First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logLData
    y = logPData
    data = pandas.DataFrame({'logL': x0, 'logP': y})
    ## Fit the model
    model = wls("logP ~ logL", data).fit()
    ## Save the model results into variables
    V = model._results.params[0]
    WRatio = model._results.params[1]
    print('W=',WRatio)
    
    # Visualize the fit
    VWFit_fig = plt.figure()
    logPdivLWData = np.log10((DataToFit[PIndex]/DataToFit[LIndex]**WRatio)) # log (P/(L^W))
    print('logP=', V, '+', WRatio, 'logL')
    plot.scatter(logLData,logPData,s=2,color='blue')
    coordinate_list = np.asarray([3,3.5,4])
    logLvlogP_predict=V + WRatio *coordinate_list
    plot.plot(coordinate_list,logLvlogP_predict, color='black')
    plot.xlabel("logL")
    plot.ylabel("logP")
    plot.xlim(coordinate_list[0], coordinate_list[-1])
    plot.ylim(1.7, 3)
    plot.title(specify_star_type+ ' - logP = V + W logL')
    if save_figures_VW != 0:
        VWFit_fig.savefig(save_directory+'VWFit.png')
    plot.close()
    
    return V, WRatio



In [10]:
# Functions to sort a list
def examine_index_Qx(elem): 
    # Sorting Key. Will sort a list based on the specified index hard coded here.
    return elem[MdotperMIndex_sorting]

def sort_star_data_Qx(DataToSort):
    TransposedArray=np.transpose(DataToSort)
    TempList=TransposedArray.tolist()
    TempList=sorted(TempList,key = operator.itemgetter(MdotperMIndex_sorting))
    return np.transpose(np.asarray(TempList))

def sort_star_data_Qx_pw(DataToSort):
    TransposedArray=np.transpose(DataToSort)
    TempList=TransposedArray.tolist()
    TempList=sorted(TempList,key = operator.itemgetter(-1))
    return np.transpose(np.asarray(TempList))

def MassRangeSelect_Qx (DataToTrim, LowerBound, UpperBound ):
    
    return np.transpose(np.transpose(sort_star_data_Qx(DataToTrim))[LowerBound-1:UpperBound-1])

In [11]:
def VWFit_Qx_method (data_set, specify_star_type ='', VW_fit_lower_percentile_bound=0, VW_fit_upper_percentile_bound = 100, save_figures_VW = 0, method_folder = 'Percentile_Range'): 
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/C1/'
    else:
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/Other/'
    
    # First, set up data for the initial VW Fit
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    PercentileLower= int(np.floor(VW_fit_lower_percentile_bound/100*len(np.transpose(DataToFit))))
    PercentileUpper = int(np.ceil(VW_fit_upper_percentile_bound/100*len(np.transpose(DataToFit))))
    #print(VW_fit_lower_percentile_bound, 'th Percentile:', PercentileLower,VW_fit_upper_percentile_bound, 'th Percentile:', PercentileUpper)
    
    
    
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    #print('DEBUG: logLData:', logLData)
    logPData = np.log10(DataToFit[PIndex])
    #print('DEBUG: logPData:', logPData)
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotperMData = np.log10(DataToFit[MdotIndex]/DataToFit[MIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    
    DataToFit = np.stack((logMData, logRData, logLData, logPData, logMdotData, logTeffObsvData, logTeffCalcData, logMdotperMData))
    #print('DEBUG: DataToFit:', DataToFit)
    if PercentileLower !=0 and PercentileUpper != 100:
        DataToFit = MassRangeSelect_Qx (DataToFit, PercentileLower, PercentileUpper )
    print('DEBUG: DataToFit:', DataToFit)
    # Now take the columns we want from the data set and assign them to their own variables.
    logMData = DataToFit[MIndex_sorting]
    logRData = DataToFit[RIndex_sorting]
    logLData = DataToFit[LIndex_sorting]
    #print('DEBUG: logLData:', logLData)
    logPData = DataToFit[PIndex_sorting]
    #print('DEBUG: logPData:', logPData)
    logMdotperMData = DataToFit[MdotperMIndex_sorting]
    logTeffObsvData = DataToFit[TeffObsvIndex_sorting]
    logTeffCalcData = DataToFit[TeffCalcIndex_sorting]
    
    
    # Next, do the fit to obtain V and W
    ## logP ~ V + W log L
    ## First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logLData
    y = logPData
    data = pandas.DataFrame({'logL': x0, 'logP': y})
    ## Fit the model
    model = wls("logP ~ logL", data).fit()
    ## Save the model results into variables
    V = model._results.params[0]
    WRatio = model._results.params[1]
    print('W=',WRatio)
    
    # Visualize the fit
    VWFit_fig = plt.figure()
    logPdivLWData = np.log10((DataToFit[PIndex_sorting]/DataToFit[LIndex_sorting]**WRatio)) # log (P/(L^W))
    print('logP=', V, '+', WRatio, 'logL')
    plot.scatter(logLData,logPData,s=2,color='blue')
    coordinate_list = np.asarray([3,3.5,4])
    logLvlogP_predict=V + WRatio *coordinate_list
    plot.plot(coordinate_list,logLvlogP_predict, color='black')
    plot.xlabel("logL")
    plot.ylabel("logP")
    plot.xlim(coordinate_list[0], coordinate_list[-1])
    plot.ylim(1.7, 3)
    plot.title(specify_star_type+ ' - logP = V + W logL')
    if save_figures_VW != 0:
        VWFit_fig.savefig(save_directory+'VWFit.png')
    plot.close()
    
    return V, WRatio

In [12]:
# This function takes a set of stars (already sorted by type and pulsation mode) and fits a mass loss fit using the stars in a given percentile range as the basis for the fit
def mdot_fit_Q_percentile_select (data_set, specify_star_type ='', VW_fit_lower_percentile_bound=0, VW_fit_upper_percentile_bound = 100, save_figures = 0, method_folder = 'Percentile_Range'):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/C1/'
    else:
        save_directory = 'MDot_Fits/Q_method/'+method_folder+'/Other/'
    # Create directories if they do not exist
    
    # Use an algorithm to determine V and W. Defined earlier in the code.
    if method_folder == 'Percentile_Range':
        V, WRatio = VWFit_Q_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures)
    else:
        V, WRatio = VWFit_Q_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures, method_folder = method_folder)
    # Restore Unrestricted data so we can plot all of it, not just the data used to make our formula
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    ## Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    logPdivLWData = np.log10((DataToFit[PIndex]/DataToFit[LIndex]**WRatio)) # log (P/(L^W))
    
    # Do our first fit for determining a function for Mdot
    ## logMdot ~ logA + c log((P/L)^W) + d logM
    x0 = logPdivLWData
    x1 = logMData
    y = logMdotData
    data = pandas.DataFrame({'logPdivLW': x0, 'logM': x1, 'logMdot': y})
    # Fit the model
    model = wls("logMdot ~ logPdivLW + logM", data).fit()
    
    # Refine the previous fit using an ODR Method
    ## Define a function (four-variable linear in our case) to fit the data with.
    def fitting_func(B, x):
        return B[0]*x[0] + B[1]*x[1] + B[2] # A simple linear fit
    ## Create a model for fitting.
    linear_model = Model(fitting_func)
    # Create a RealData object using our initiated data from above.
    data = RealData((logPdivLWData, logMData), y=logMdotData) # Data: x=logL, y=logR
    ## Set up ODR with the model and data.
    odr = ODR(data, linear_model, beta0=[model._results.params[1],model._results.params[2],model._results.params[0]]) # Use the linear regression as an initial guess
    ## Run the regression.
    outPdivLW_M = odr.run()
    
    # Using the above fit, calculate the resutls for logMdot
    logMdot_PdivLM = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2]
    print( 'logMdot =', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM + ', outPdivLW_M.beta[2])
    
    # Do a fit of calculated Mdot vs observed Mdot for plotting
    x0 = logMdotData
    y = logMdot_PdivLM
    data = pandas.DataFrame({'logMdotData': x0, 'logMdot_PdivLM': y})
    # Fit the model
    model = ols("logMdot_PdivLM ~ logMdotData", data).fit()
    PdivLMLine = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    
    # Plot our initial fit, and see how it compares to observation
    first_fit_mdot_fig = plt.figure()
    SlopeTestNoErrors=PdivLMLine[0] + PdivLMLine[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    plot.scatter(logMdotData, logMdot_PdivLM, s=2, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        first_fit_mdot_fig.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    # With that first fit done, we will start improving it by subtracting the residuals from the fit
    ResidualYAxis = logMdot_PdivLM - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    PlotTitle = specify_star_type + ' - No Errors Included' # Label the plot with whatever data set we are using
    
    # Fit logP to the residuals in a linear fit
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logPData
    y = ResidualYAxis
    data = pandas.DataFrame({'logPData': x0, 'ResidualYAxis': y})
    model = ols("ResidualYAxis ~ logPData", data).fit()
    ResidualsFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residual Fit: logMdotCalc-logMdotObsv =', ResidualsFit[0], '+', ResidualsFit[1], 'logP')
    
    # Now, we are going to start comparing the residual to various useful parameters
    ylabel = 'Mdot Residuals' # What is on the y label until we change this value
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    first_resid_vs_P = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_P.savefig(save_directory+'Residual_Images/first_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    first_resid_vs_L = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    plot.title(PlotTitle)
    if save_figures != 0:
        first_resid_vs_L.savefig(save_directory+'Residual_Images/first_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    first_resid_vs_Q = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_Q.savefig(save_directory+'Residual_Images/first_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    first_resid_vs_M = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_M.savefig(save_directory+'Residual_Images/first_resid_vs_logM.png')
    plot.close()
    
    # Now, remove the residuals
    NewMdotCalc = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2] - ResidualsFit[0] - ResidualsFit[1]*logPData
    ylabel = "NewlogMdotCalc"
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM - ', ResidualsFit[1], 'logP', '; W =', WRatio )
    # Simplified statement: constants + A log P + B log L + log M
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+ (', outPdivLW_M.beta[0] - ResidualsFit[1],') logP + (', -1*outPdivLW_M.beta[0]*WRatio,') logL + ', outPdivLW_M.beta[1], 'logM'  )
    
    # logMdot ~ logM + logL + logP
    xdataSet = logMdotData
    x0 = xdataSet
    y = NewMdotCalc
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('NewMdotCalc Fit: NewlogMdotCalc =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logMdot_obsv')
    newmdotcalc_vs_mdotobsv = plt.figure()
    plot.scatter(xdataSet, NewMdotCalc, s=2, color='green')
    plot.plot(np.asarray([-9,-7,-6,-4]),np.asarray([-9,-7,-6,-4]),color='black')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.xlim(mdot_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logMdot_obsv")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        newmdotcalc_vs_mdotobsv.savefig(save_directory+'newmdotcalc_vs_mdotobsv.png')
    plot.close()
    
    # Now, we're going to compare residuals
    
    NewResidual = NewMdotCalc - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    ylabel = 'Old Residual' # What is on the y label until we change this value
    xdataSet = NewResidual
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residuals Fit: OldResidual =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'NewResidual')
    second_resid_vs_first_resid = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-4,-1,0,1,4]),np.asarray([-4,-1,0,1,4]),color='black')
    plot.xlim(resid_limits)
    plot.ylim(resid_limits)
    plot.xlabel("New Residual")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_first_resid.savefig(save_directory+'Residual_Images/second_resid_vs_first_resid.png')
    plot.close()
    
    ### RMS of Data
    RMSE = calc_rms_error (NewMdotCalc, logMdotData)
    print ('RMS Error:', RMSE)
    print ('STD Dev:', np.std(NewMdotCalc))
    
    ylabel = 'Second Residual'
    ResidualYAxis = NewResidual
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    second_resid_vs_logP = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logP.savefig(save_directory+'Residual_Images/second_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    second_resid_vs_logL = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logL.savefig(save_directory+'Residual_Images/second_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    second_resid_vs_logQ = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logQ.savefig(save_directory+'Residual_Images/second_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    second_resid_vs_logM = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logM.savefig(save_directory+'Residual_Images/second_resid_vs_logM.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logR
    xdataSet = logRData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logR')
    second_resid_vs_logR = plt.figure()
    plot.scatter(logRData, NewMdotCalc, s=2, color='green')
    plot.xlim(radius_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logR")
    plot.ylabel('Mdot_calc_New')
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logR.savefig(save_directory+'Residual_Images/second_resid_vs_logR.png')
    plot.close()
    
    #return 1 # Placeholder. Should change to return values of interest in an array.
    return [-ResidualsFit[0] + outPdivLW_M.beta[2], outPdivLW_M.beta[0] - ResidualsFit[1], -1*outPdivLW_M.beta[0]*WRatio, outPdivLW_M.beta[1] ], NewMdotCalc, logMdotData, [V,WRatio]
    # What data do we need to carry over for a piecewise?: RawMdotData, NewMdotcalc coefficients, NewMdotCalc of input data

In [13]:
mdot_fit_Q_percentile_select (M0StarData, specify_star_type ='M0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

W= 0.6830057183526329
logP= -0.21758717714637626 + 0.6830057183526329 logL
logMdot = 14.216860652255718 log(P/L^W) +  7.489751883786768 logM +  -3.816144909387197
Comparison to Observed: logMdot_calc = ( -0.12539009253942016 ) + ( 0.9814952846729118 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -12.751897087121566 + 5.810513694459377 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -12.751897087121566 + 5.810513694459377 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -12.180592895403855 + 3.453160001241986 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 1.2115958697518312 + 5.6460462138907666 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.018588285133237306 + 1.526553717102509 logM
NewMdotCalc = 8.935752177734368 + 14.216860652255718 log(P/L^W) +  7.489751883786768 logM -  5.810513694459377 logP ; W = 0.6830057183526329
NewMdotCalc = 8.935752177734368 + ( 8.406346957796341 ) logP + ( -9.710197122513199 ) logL +  7.48975188378

([8.935752177734368, 8.406346957796341, -9.710197122513199, 7.489751883786768],
 array([-6.802976  , -6.64255655, -6.84202646, ..., -6.98887097,
        -6.92851287, -6.99324967]),
 array([-6.54925578, -6.93120769, -6.48855794, ..., -6.54925578,
        -6.59501328, -7.22703649]),
 [-0.21758717714637626, 0.6830057183526329])

In [14]:
mdot_fit_Q_percentile_select (M1StarData, specify_star_type ='M1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

W= 0.6117650063628665
logP= -0.24028083894089158 + 0.6117650063628665 logL
logMdot = 14.563637932306683 log(P/L^W) +  4.1232430418830495 logM +  -3.3070239796311416
Comparison to Observed: logMdot_calc = ( -0.06049024134747425 ) + ( 0.9912363877676534 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -5.568055710028659 + 2.865935510095914 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -5.568055710028659 + 2.865935510095914 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -4.758705646310228 + 1.3284496676312563 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 1.9031453473892246 + 7.6556575676825505 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.0026555796100458814 + 0.4377481577243102 logM
NewMdotCalc = 2.2610317303975176 + 14.563637932306683 log(P/L^W) +  4.1232430418830495 logM -  2.865935510095914 logP ; W = 0.6117650063628665
NewMdotCalc = 2.2610317303975176 + ( 11.697702422210769 ) logP + ( -8.909524052324082 ) logL +  4.12324

([2.2610317303975176,
  11.697702422210769,
  -8.909524052324082,
  4.1232430418830495],
 array([-7.09262573, -6.95275189, -6.92308815, ..., -7.49833835,
        -7.66164392, -7.53564827]),
 array([-7.28208406, -6.98696292, -7.01143669, ..., -7.45234577,
        -6.6461658 , -7.40312775]),
 [-0.24028083894089158, 0.6117650063628665])

In [15]:
mdot_fit_Q_percentile_select (C0StarData, specify_star_type ='C0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

W= 0.41870953488462537
logP= 0.9221255886346236 + 0.41870953488462537 logL
logMdot = 9.571964457927034 log(P/L^W) +  2.4178993691909967 logM +  -15.784237799172608
Comparison to Observed: logMdot_calc = ( -0.09547095235283634 ) + ( 0.9852941326157518 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -9.825574748121028 + 3.8971993666460527 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -9.825574748121028 + 3.8971993666460527 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -4.869695401592185 + 1.2801243843783556 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -4.0997857167148775 + 4.416044615556374 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.07405406497256747 + 0.44129722191556786 logM
NewMdotCalc = -5.95866305105158 + 9.571964457927034 log(P/L^W) +  2.4178993691909967 logM -  3.8971993666460527 logP ; W = 0.41870953488462537
NewMdotCalc = -5.95866305105158 + ( 5.674765091280982 ) logP + ( -4.007872786110794 ) logL +  2.4178993

([-5.95866305105158,
  5.674765091280982,
  -4.007872786110794,
  2.4178993691909967],
 array([-6.96387312, -6.86479051, -6.47775163, ..., -6.60923857,
        -6.83659074, -7.05356406]),
 array([-6.74122305, -6.74122305, -5.43505341, ..., -5.20310433,
        -6.08571873, -7.04225304]),
 [0.9221255886346236, 0.41870953488462537])

In [16]:
mdot_fit_Q_percentile_select (C1StarData, specify_star_type ='C1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

W= 0.37085618916281626
logP= 0.8002687209340122 + 0.37085618916281626 logL
logMdot = 7.142007947003209 log(P/L^W) +  2.403684298376941 logM +  -12.753237599236027
Comparison to Observed: logMdot_calc = ( -0.1795794608954967 ) + ( 0.9730970128239868 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -6.589034994096469 + 2.955133692683276 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -6.589034994096469 + 2.955133692683276 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -5.458287658454416 + 1.411163676795371 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -2.633358394072414 + 3.311384956683115 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.07773332685161935 + 0.4688460192885994 logM
NewMdotCalc = -6.1642026051395575 + 7.142007947003209 log(P/L^W) +  2.403684298376941 logM -  2.955133692683276 logP ; W = 0.37085618916281626
NewMdotCalc = -6.1642026051395575 + ( 4.186874254319933 ) logP + ( -2.648657850196159 ) logL +  2.403684298376

([-6.1642026051395575,
  4.186874254319933,
  -2.648657850196159,
  2.403684298376941],
 array([-6.82589386, -7.02222285, -6.58682811, -6.20231344, -6.70545201,
        -6.149154  , -6.88807124, -6.29914664, -7.06656597, -6.60850975,
        -6.82470499, -6.33461089, -6.86908805, -6.7056402 , -6.84428138,
        -6.87961088, -6.76475215, -6.7098287 , -6.98134079, -6.90718051,
        -6.97978038, -6.56358537, -6.70586387, -6.68594051, -6.78386852,
        -6.24039703, -6.63623245, -6.93243769, -6.86333246, -6.60242063,
        -6.66651118, -6.46028422, -6.77031438, -6.54516472, -6.53124348,
        -6.48520697, -6.84154419, -6.88669956, -6.77073092, -6.47823696,
        -6.74750532, -6.74058761, -6.59653492, -6.51996864, -6.67637262,
        -6.56756318, -6.35824911, -6.53369439, -6.5598103 , -6.49751467,
        -6.88407599, -6.67613471, -6.63181842, -6.63766558, -6.47603899,
        -6.76847757, -6.37131939, -6.76838927, -6.83520454, -6.74448186,
        -6.57603228, -6.74395073, -6

In [17]:
def mdot_fit_Q_piecewise (data_set_pw, specify_star_type ='', number_of_ranges = 2, save_figures = 0 ):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Q_method/Sectioned/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Q_method/Sectioned/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Q_method/Sectioned/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Q_method/Sectioned/C1/'
    else:
        save_directory = 'MDot_Fits/Q_method/Sectioned/Other/'
    
    
    DataToSplit = sort_star_data_Q (data_set_pw )
    #print('Debug: DataToFit:')
    data_for_sectioning = np.transpose(DataToSplit)
    #print(len(data_for_sectioning))
    # Determine split points
    split_increment = int(np.floor(len(data_for_sectioning)/number_of_ranges))
    if split_increment < 1:
        print ('ERROR: More desired ranges than Data Points')
    splitting_points = []
    split_point = split_increment # Initialize this at the lowest split point
    while split_point < number_of_ranges*split_increment:
        splitting_points.append(split_point)
        split_point = split_point + split_increment
    print('Splitting Points:', splitting_points)
    # Take data set and split into n ranges. Defaults to two.
    sectioned_data = np.transpose(np.split (data_for_sectioning, splitting_points))
    partition_check (DataToSplit,sectioned_data)
    
    #print(sectioned_data[1])
    print('Data pts. in array:', len(sectioned_data[1]))
    returned_fit_parameters = []
    returned_Mdot_obsv_Data = []
    returned_Mdot_calc_Data = []
    returned_fit_Ws = []
    for i in range(0, number_of_ranges):
        print('*Range ',i, 'Start*')
        working_data = np.transpose(np.asarray(sectioned_data[i]))
        results = mdot_fit_Q_percentile_select (working_data, specify_star_type, method_folder = 'Sectioned')
        returned_fit_parameters.append(results[0])
        returned_Mdot_calc_Data.append(results[1])
        returned_Mdot_obsv_Data.append(results[2])
        returned_fit_Ws.append(results[3][1])
        print('*Range',i, 'End*\n')
        
    logMdotData = np.log10(data_set_pw[MdotIndex])
    calc_MDot_Data = np.concatenate(returned_Mdot_calc_Data)
    returned_stars_obsv = 0
    for i in range(0,len(returned_Mdot_obsv_Data)):
        returned_stars_obsv = returned_stars_obsv + len(returned_Mdot_obsv_Data[i])
    returned_stars_calc = 0
    for i in range(0,len(returned_Mdot_calc_Data)):
        returned_stars_calc = returned_stars_calc + len(returned_Mdot_calc_Data[i])
    #print('DEBUG: len(data_for_sectioning)=',len(data_for_sectioning), 'returned_stars_obsv=',returned_stars_obsv, 'len(returned_stars_calc)=', returned_stars_calc)
    mdot_calc_vs_obsv = plt.figure()
    #print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    legend_labels = ['1:1 Line']
    for i in range (0, number_of_ranges):
        plot.scatter(returned_Mdot_obsv_Data[i], returned_Mdot_calc_Data[i], s=2)
        legend_labels.append('Range ' + str(i) + '(' + str(np.min(returned_Mdot_obsv_Data[i]))[:5] + ' -- ' + str(np.max(returned_Mdot_obsv_Data[i]))[:5]+')')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.legend(legend_labels)
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        mdot_calc_vs_obsv.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    print('W Values:', returned_fit_Ws)

In [18]:
mdot_fit_Q_piecewise (M0StarData, specify_star_type ='M0', number_of_ranges = 10, save_figures = 1 )

Splitting Points: [195, 390, 585, 780, 975, 1170, 1365, 1560, 1755]


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:8: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  


Checked: Number of sources match after partition.
Checked: All sources accounted for, in original order.
Data pts. in array: 195
*Range  0 Start*
W= 0.6902047972018746
logP= -0.2994665383608963 + 0.6902047972018746 logL
logMdot = 15.699343500070123 log(P/L^W) +  10.157387405120078 logM +  -3.0500172957076743
Comparison to Observed: logMdot_calc = ( -0.6292365339956456 ) + ( 0.9162554458142159 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -16.1837188954577 + 7.73871886780436 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -16.1837188954577 + 7.73871886780436 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -19.06951589312227 + 5.50537197919524 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 2.200273226717725 + 7.347315198144859 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.06759661071571496 + 2.888401396889228 logM
NewMdotCalc = 13.133701599750028 + 15.699343500070123 log(P/L^W) +  10.157387405120078 logM -  7.73871886780436 l

NewMdotCalc = -6.6290114378963345 + 0.2674116363547611 log(P/L^W) +  0.15092195916735357 logM -  0.038439957842916574 logP ; W = 0.7049029937934523
NewMdotCalc = -6.6290114378963345 + ( 0.22897167851184455 ) logP + ( -0.1884992630416771 ) logL +  0.15092195916735357 logM
NewMdotCalc Fit: NewlogMdotCalc = -5.909105976010403 + 0.12979173142283584 logMdot_obsv
Residuals Fit: OldResidual = 2.7031145145563163e-08 + 0.9999999999999994 NewResidual
RMS Error: 0.029548615065148106
STD Dev: 0.01141162279723713
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -5.468715758016884e-16 + 2.662800535624399e-16 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -8.170217594680174e-06 + 2.3093555146333922e-06 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -1.752846512236277e-06 + -5.75198432600614e-06 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -1.5168129624795247e-07 + 5.448713120295473e-06 logM
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -6.631728

RMS Error: 0.16989548303142463
STD Dev: 0.06297038902012829
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 5.4817261840867104e-15 + 1.0408340855860843e-16 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.5319658268203699 + -0.15160450591869884 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.17926468549226146 + 0.2738959194133147 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.012754503215653737 + -0.23947197552868227 logM
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.5219149413910187 + -0.23814634815280777 logR
*Range 9 End*

W Values: [0.6902047972018746, 0.6428977521904342, 0.6683298309922591, 0.5839452037657368, 0.7049029937934523, 0.6181222303075726, 0.5836364956305451, 0.5670284912216477, 0.5352051968599562, 0.45559029716628224]


In [19]:
mdot_fit_Q_piecewise (M1StarData, specify_star_type ='M1', number_of_ranges = 10, save_figures = 1 )

Splitting Points: [216, 432, 648, 864, 1080, 1296, 1512, 1728, 1944]


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:8: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  


Checked: Number of sources match after partition.
Checked: All sources accounted for, in original order.
Data pts. in array: 216
*Range  0 Start*
W= 0.6596587680982428
logP= -0.4490851924938601 + 0.6596587680982428 logL
logMdot = 12.461503164637872 log(P/L^W) +  4.68935811853503 logM +  -1.8539816047516573
Comparison to Observed: logMdot_calc = ( -0.5032392880386065 ) + ( 0.9346012277787933 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -6.40093170176471 + 3.58550538228378 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -6.40093170176471 + 3.58550538228378 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -6.480677667098554 + 1.9133587655409643 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 3.1900077216835916 + 7.103346460745372 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.038074620597664185 + 0.7297198114139751 logM
NewMdotCalc = 4.5469500970130525 + 12.461503164637872 log(P/L^W) +  4.68935811853503 logM -  3.58550538228378 l

Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.0029773031199612637 + -0.008921015377193992 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 1.9349597523245124e-05 + 0.002902677245992232 logM
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.0019546556022449757 + 0.0008769634442898811 logR
*Range 4 End*

*Range  5 Start*
W= 0.5761092929785699
logP= -0.1136857817513216 + 0.5761092929785699 logL
logMdot = 0.2852238021207478 log(P/L^W) +  0.12142752460925381 logM +  -6.7909335142325515
Comparison to Observed: logMdot_calc = ( -6.40758278100712 ) + ( 0.06093294850442166 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -0.1399423325389467 + 0.07169511785049926 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.1399423325389467 + 0.07169511785049926 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.11644560727783217 + 0.0324775226482245 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.014296041647215495 + 0.125750564141

Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.1255048252453922 + -2.1795828499514927 logM
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 2.7154732731943394 + -1.166751276726136 logR
*Range 9 End*

W Values: [0.6596587680982428, 0.6204255597312601, 0.6687543840464056, 0.5938982764159155, 0.6381904592926364, 0.5761092929785699, 0.5362881505672725, 0.532304373655268, 0.47977797241139974, 0.5088718957910813]


In [20]:
mdot_fit_Q_piecewise (C0StarData, specify_star_type ='C0', number_of_ranges = 10, save_figures = 1 )

Splitting Points: [199, 398, 597, 796, 995, 1194, 1393, 1592, 1791]


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:8: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  


Checked: Number of sources match after partition.
Checked: All sources accounted for, in original order.
Data pts. in array: 199
*Range  0 Start*
W= 0.3515945294057093
logP= 1.112398735712215 + 0.3515945294057093 logL
logMdot = 15.818507780222326 log(P/L^W) +  5.733830297623777 logM +  -25.755319439780937
Comparison to Observed: logMdot_calc = ( -0.8355854190862646 ) + ( 0.8821590675931664 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -28.729225597087428 + 11.788280538762388 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -28.729225597087428 + 11.788280538762388 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -15.467561488132231 + 4.105308629140364 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -13.173809286343477 + 11.842704314988366 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.32133819412529196 + 1.7251211066966008 logM
NewMdotCalc = 2.973906157306491 + 15.818507780222326 log(P/L^W) +  5.733830297623777 logM -  11.788280

Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.007834271912220322 + -0.002036010462650997 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.003929252731987069 + -0.004444272011909871 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.0003942160438585652 + -0.0019573294821527056 logM
NewMdotCalc = -6.538550685544018 + -0.027749931230522284 log(P/L^W) +  -0.047178318408643415 logM -  -0.004620463003107062 logP ; W = 0.4287598761628081
NewMdotCalc = -6.538550685544018 + ( -0.023129468227415222 ) logP + ( 0.011898057077925175 ) logL +  -0.047178318408643415 logM
NewMdotCalc Fit: NewlogMdotCalc = -6.293715481048341 + 0.04072067508642929 logMdot_obsv
Residuals Fit: OldResidual = 3.2608421437216664e-10 + 0.9999999999999982 NewResidual
RMS Error: 0.026453982905508544
STD Dev: 0.005455553650434962
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -4.440892098500626e-16 + 2.194425197110661e-16 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.000

Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 3.5222441497012595 + -1.4379405833599488 logR
*Range 9 End*

W Values: [0.3515945294057093, 0.38442019223741153, 0.2871343969007738, 0.4071789229960787, 0.40007088075379454, 0.4287598761628081, 0.3182461117994602, 0.36892961780816885, 0.3800465517007703, 0.3216935964432605]


In [21]:
mdot_fit_Q_piecewise (C1StarData, specify_star_type ='C1', number_of_ranges = 10, save_figures = 1 )

Splitting Points: [78, 156, 234, 312, 390, 468, 546, 624, 702]
Checked: Number of sources match after partition.
Checked: All sources accounted for, in original order.
Data pts. in array: 78
*Range  0 Start*
W= 0.3574911927636123
logP= 0.7862289400785089 + 0.3574911927636123 logL
logMdot = 109.64686044730423 log(P/L^W) +  30.106238575515924 logM +  -98.23190010807208
Comparison to Observed: logMdot_calc = ( -5.3614558633917175 ) + ( 0.25230703523060005 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv =

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:8: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  


 -156.09204682789877 + 72.95559251570278 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -156.09204682789877 + 72.95559251570278 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -82.90490127223893 + 21.900050389944077 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -64.55514739118317 + 82.10731531587678 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.9140515528559721 + 5.669624563582879 logM
NewMdotCalc = 57.86014671982669 + 109.64686044730423 log(P/L^W) +  30.106238575515924 logM -  72.95559251570278 logP ; W = 0.3574911927636123
NewMdotCalc = 57.86014671982669 + ( 36.69126793160146 ) logP + ( -39.197786924092135 ) logL +  30.106238575515924 logM
NewMdotCalc Fit: NewlogMdotCalc = -2.2824780994661036 + 0.68169228132654 logMdot_obsv
Residuals Fit: OldResidual = 1.0231100816548988e-07 + 1.0000000000000366 NewResidual
RMS Error: 0.5628976232589423
STD Dev: 0.5707895778149973
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -3.2696068

Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -5.06729978864493e-15 + 1.3109833169619971e-15 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -3.5935470978317534e-15 + 3.2159878561102488e-15 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -6.317587051815579e-17 + 6.984089516575774e-16 logM
NewMdotCalc = -6.624109646181172 + 7.771561172376096e-15 log(P/L^W) +  9.645062526431047e-16 logM -  3.6421594264769094e-15 logP ; W = 0.2903443325978429
NewMdotCalc = -6.624109646181172 + ( 4.129401745899186e-15 ) logP + ( -2.2564287418368473e-15 ) logL +  9.645062526431047e-16 logM
NewMdotCalc Fit: NewlogMdotCalc = -0.1475998784312682 + 0.9777177784917321 logMdot_obsv
Residuals Fit: OldResidual = 5.693451408334134e-17 + -3.241538893906194e-33 NewResidual
RMS Error: 2.24873312656119e-16
STD Dev: 9.698272657828412e-16
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 1.4551188894765017e-15 + -6.661110923909827e-16 logP
Resid. vs Parameter Fit: logMdot_calc - logMd

KeyboardInterrupt: 

In [ ]:
# This function takes a set of stars (already sorted by type and pulsation mode) and fits a mass loss fit using the stars in a given percentile range as the basis for the fit
def mdot_fit_Qx_percentile_select (data_set, specify_star_type ='', VW_fit_lower_percentile_bound=0, VW_fit_upper_percentile_bound = 100, save_figures = 0, method_folder = 'Percentile_Range'):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/C1/'
    else:
        save_directory = 'MDot_Fits/Qx_method/'+method_folder+'/Other/'
    # Create directories if they do not exist
    
    # Use an algorithm to determine V and W. Defined earlier in the code.
    if method_folder == 'Percentile_Range':
        V, WRatio = VWFit_Qx_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures)
    else:
        V, WRatio = VWFit_Qx_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures, method_folder = method_folder)
   
    
    # Use an algorithm to determine V and W. Defined earlier in the code.
    V, WRatio = VWFit_Qx_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures)
    
    # Restore Unrestricted data so we can plot all of it, not just the data used to make our formula
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    ## Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    logPdivLWData = np.log10((DataToFit[PIndex]/DataToFit[LIndex]**WRatio)) # log (P/(L^W))
    
    # Do our first fit for determining a function for Mdot
    ## logMdot ~ logA + c log((P/L)^W) + d logM
    x0 = logPdivLWData
    x1 = logMData
    y = logMdotData
    data = pandas.DataFrame({'logPdivLW': x0, 'logM': x1, 'logMdot': y})
    # Fit the model
    model = wls("logMdot ~ logPdivLW + logM", data).fit()
    
    # Refine the previous fit using an ODR Method
    ## Define a function (four-variable linear in our case) to fit the data with.
    def fitting_func(B, x):
        return B[0]*x[0] + B[1]*x[1] + B[2] # A simple linear fit
    ## Create a model for fitting.
    linear_model = Model(fitting_func)
    # Create a RealData object using our initiated data from above.
    data = RealData((logPdivLWData, logMData), y=logMdotData) # Data: x=logL, y=logR
    ## Set up ODR with the model and data.
    odr = ODR(data, linear_model, beta0=[model._results.params[1],model._results.params[2],model._results.params[0]]) # Use the linear regression as an initial guess
    ## Run the regression.
    outPdivLW_M = odr.run()
    
    # Using the above fit, calculate the resutls for logMdot
    logMdot_PdivLM = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2]
    print( 'logMdot =', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM + ', outPdivLW_M.beta[2])
    
    # Do a fit of calculated Mdot vs observed Mdot for plotting
    x0 = logMdotData
    y = logMdot_PdivLM
    data = pandas.DataFrame({'logMdotData': x0, 'logMdot_PdivLM': y})
    # Fit the model
    model = ols("logMdot_PdivLM ~ logMdotData", data).fit()
    PdivLMLine = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    
    # Plot our initial fit, and see how it compares to observation
    first_fit_mdot_fig = plt.figure()
    SlopeTestNoErrors=PdivLMLine[0] + PdivLMLine[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    plot.scatter(logMdotData, logMdot_PdivLM, s=2, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        first_fit_mdot_fig.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    # With that first fit done, we will start improving it by subtracting the residuals from the fit
    ResidualYAxis = logMdot_PdivLM - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    PlotTitle = specify_star_type + ' - No Errors Included' # Label the plot with whatever data set we are using
    
    # Fit logP to the residuals in a linear fit
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logPData
    y = ResidualYAxis
    data = pandas.DataFrame({'logPData': x0, 'ResidualYAxis': y})
    model = ols("ResidualYAxis ~ logPData", data).fit()
    ResidualsFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residual Fit: logMdotCalc-logMdotObsv =', ResidualsFit[0], '+', ResidualsFit[1], 'logP')
    
    # Now, we are going to start comparing the residual to various useful parameters
    ylabel = 'Mdot Residuals' # What is on the y label until we change this value
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    first_resid_vs_P = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_P.savefig(save_directory+'Residual_Images/first_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    first_resid_vs_L = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    plot.title(PlotTitle)
    if save_figures != 0:
        first_resid_vs_L.savefig(save_directory+'Residual_Images/first_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    first_resid_vs_Q = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_Q.savefig(save_directory+'Residual_Images/first_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    first_resid_vs_M = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_M.savefig(save_directory+'Residual_Images/first_resid_vs_logM.png')
    plot.close()
    
    # Now, remove the residuals
    NewMdotCalc = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2] - ResidualsFit[0] - ResidualsFit[1]*logPData
    ylabel = "NewlogMdotCalc"
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM - ', ResidualsFit[1], 'logP', '; W =', WRatio )
    # Simplified statement: constants + A log P + B log L + log M
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+ (', outPdivLW_M.beta[0] - ResidualsFit[1],') logP + (', -1*outPdivLW_M.beta[0]*WRatio,') logL + ', outPdivLW_M.beta[1], 'logM'  )
    
    # logMdot ~ logM + logL + logP
    xdataSet = logMdotData
    x0 = xdataSet
    y = NewMdotCalc
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('NewMdotCalc Fit: NewlogMdotCalc =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logMdot_obsv')
    newmdotcalc_vs_mdotobsv = plt.figure()
    plot.scatter(xdataSet, NewMdotCalc, s=2, color='green')
    plot.plot(np.asarray([-9,-7,-6,-4]),np.asarray([-9,-7,-6,-4]),color='black')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.xlim(mdot_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logMdot_obsv")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        newmdotcalc_vs_mdotobsv.savefig(save_directory+'newmdotcalc_vs_mdotobsv.png')
    plot.close()
    
    # Now, we're going to compare residuals
    
    NewResidual = NewMdotCalc - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    ylabel = 'Old Residual' # What is on the y label until we change this value
    xdataSet = NewResidual
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residuals Fit: OldResidual =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'NewResidual')
    second_resid_vs_first_resid = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-4,-1,0,1,4]),np.asarray([-4,-1,0,1,4]),color='black')
    plot.xlim(resid_limits)
    plot.ylim(resid_limits)
    plot.xlabel("New Residual")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_first_resid.savefig(save_directory+'Residual_Images/second_resid_vs_first_resid.png')
    plot.close()
    
    ### RMS of Data
    RMSE = calc_rms_error (NewMdotCalc, logMdotData)
    print ('RMS Error:', RMSE)
    print ('STD Dev:', np.std(NewMdotCalc))
    
    ylabel = 'Second Residual'
    ResidualYAxis = NewResidual
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    second_resid_vs_logP = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logP.savefig(save_directory+'Residual_Images/second_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    second_resid_vs_logL = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logL.savefig(save_directory+'Residual_Images/second_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    second_resid_vs_logQ = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logQ.savefig(save_directory+'Residual_Images/second_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    second_resid_vs_logM = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logM.savefig(save_directory+'Residual_Images/second_resid_vs_logM.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logR
    xdataSet = logRData
    x0 = xdataSet
    y = NewMdotCalc
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logR')
    second_resid_vs_logR = plt.figure()
    plot.scatter(logRData, NewMdotCalc, s=2, color='green')
    plot.xlim(1.5,3)
    plot.ylim(mdot_limits)
    plot.xlabel("logR")
    plot.ylabel('Mdot_calc_New')
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logR.savefig(save_directory+'Residual_Images/second_resid_vs_logR.png')
    plot.close()
    
    #return 1 # Placeholder. Should change to return values of interest in an array.
    return [-ResidualsFit[0] + outPdivLW_M.beta[2], outPdivLW_M.beta[0] - ResidualsFit[1], -1*outPdivLW_M.beta[0]*WRatio, outPdivLW_M.beta[1] ], NewMdotCalc, logMdotData, [V,WRatio]

In [ ]:
mdot_fit_Qx_percentile_select (M0StarData, specify_star_type ='M0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )
mdot_fit_Qx_percentile_select (M1StarData, specify_star_type ='M1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )
mdot_fit_Qx_percentile_select (C0StarData, specify_star_type ='C0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )
mdot_fit_Qx_percentile_select (C1StarData, specify_star_type ='C1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

In [ ]:
def mdot_fit_Qx_piecewise (data_set_pw, specify_star_type ='', number_of_ranges = 2, save_figures = 0 ):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qx_method/Sectioned/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qx_method/Sectioned/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qx_method/Sectioned/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qx_method/Sectioned/C1/'
    else:
        save_directory = 'MDot_Fits/Qx_method/Sectioned/Other/'
    
    #print('In Data:', data_set_pw)
    #print('len (In Data):', len(data_set_pw))
    mdot_data = np.asarray(data_set_pw[MdotIndex])
    #print('Mdot data:', mdot_data)
    #print('len Mdot data:', len(mdot_data))
    m_data = np.asarray(data_set_pw[MIndex])
    #print('M data:', m_data)
    #print('len M data:', len(m_data))
    mdot_per_m_data = mdot_data/m_data
    #print('Mdot/M data:', mdot_per_m_data)
    #print('len Mdot/M data:', len(mdot_per_m_data))
    data_set_pw_plus_sort_column = np.vstack((data_set_pw, mdot_per_m_data))
    #print('In + Sort Data:', data_set_pw_plus_sort_column)
    #print('len (In + Sort Data):', len(data_set_pw_plus_sort_column))
    DataToSplit = sort_star_data_Qx_pw (data_set_pw_plus_sort_column )
    #print('Debug: DataToFit:')
    data_for_sectioning = np.transpose(DataToSplit)
    # Determine split points
    split_increment = int(np.floor(len(data_for_sectioning)/number_of_ranges))
    if split_increment < 1:
        print ('ERROR: More desired ranges than Data Points')
    splitting_points = []
    split_point = split_increment # Initialize this at the lowest split point
    while split_point < number_of_ranges*split_increment:
        splitting_points.append(split_point)
        split_point = split_point + split_increment
    print('Splitting Points:', splitting_points)
    # Take data set and split into n ranges. Defaults to two.
    sectioned_data = np.transpose(np.split (data_for_sectioning, splitting_points))
    print('Sorted Data:', sectioned_data)
    print('len Sorted Data:', len(sectioned_data))
    print('len Sorted Data sub array:', len(sectioned_data[0]))
    print('len Sorted Data sub sub array:', len(sectioned_data[0][0]))
    log_mdot_per_m_avg = []
    for i in range (0, len(sectioned_data)):
        working_data = np.transpose(sectioned_data[i])
        mean_value = np.mean(working_data[-1])
        log_mdot_per_m_avg.append(np.log10(mean_value))
    #print(sectioned_data[1])
    print('Data pts. in array:', len(sectioned_data[1]))
    returned_fit_parameters = []
    returned_Mdot_obsv_Data = []
    returned_Mdot_calc_Data = []
    returned_fit_Ws = []
    for i in range(0, number_of_ranges):
        print('*Range ',i, 'Start*')
        working_data = np.transpose(np.asarray(sectioned_data[i]))
        results = mdot_fit_Qx_percentile_select (working_data, specify_star_type, method_folder = 'Sectioned')
        returned_fit_parameters.append(results[0])
        returned_Mdot_calc_Data.append(results[1])
        returned_Mdot_obsv_Data.append(results[2])
        returned_fit_Ws.append(results[3][1])
        print('*Range',i, 'End*\n')
        
    logMdotData = np.log10(data_set_pw[MdotIndex])
    calc_MDot_Data = np.concatenate(returned_Mdot_calc_Data)
    mdot_calc_vs_obsv = plt.figure()
    #print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    legend_labels = ['1:1 Line']
    for i in range (0, number_of_ranges):
        plot.scatter(returned_Mdot_obsv_Data[i], returned_Mdot_calc_Data[i], s=2)
        legend_labels.append(str(log_mdot_per_m_avg[i])[:5])
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.legend(legend_labels)
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        mdot_calc_vs_obsv.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    print('W Values:', returned_fit_Ws)
    

In [ ]:
mdot_fit_Qx_piecewise (M0StarData, specify_star_type ='M0', number_of_ranges = 5, save_figures = 1 )

In [ ]:
mdot_fit_Qx_piecewise (M1StarData, specify_star_type ='M1', number_of_ranges = 5, save_figures = 1 )

In [ ]:
mdot_fit_Qx_piecewise (C0StarData, specify_star_type ='C0', number_of_ranges = 5, save_figures = 1 )

In [ ]:
mdot_fit_Qx_piecewise (C1StarData, specify_star_type ='C1', number_of_ranges = 5, save_figures = 1 )

In [ ]:
def sort_star_data_MDRD(DataToSort):
    TransposedArray=np.transpose(DataToSort)
    TempList=TransposedArray.tolist()
    TempList=sorted(TempList,key = operator.itemgetter(-1))
    return np.transpose(np.asarray(TempList))

def MassRangeSelect_MDRD (DataToTrim, LowerBound, UpperBound ):
    return np.transpose(np.transpose(sort_star_data_MDRD(DataToTrim))[LowerBound-1:UpperBound-1])

In [ ]:
# This function takes a set of stars (already sorted by type and pulsation mode) and fits a mass loss fit using the stars in a given percentile range as the basis for the fit
def mdot_fit_Qz_percentile_select (data_set, specify_star_type ='', VW_fit_lower_percentile_bound=0, VW_fit_upper_percentile_bound = 100, save_figures = 0, method_folder = 'Percentile_Range'):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qz_method/'+method_folder+'/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qz_method/'+method_folder+'/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qz_method/'+method_folder+'/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qz_method/'+method_folder+'/C1/'
    else:
        save_directory = 'MDot_Fits/Qz_method/'+method_folder+'/Other/'
    # Create directories if they do not exist
    
    # Use an algorithm to determine V and W. Defined earlier in the code.
    if method_folder == 'Percentile_Range':
        V, WRatio = VWFit_Q_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures)
    else:
        V, WRatio = VWFit_Q_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures, method_folder = method_folder)
    
    # Restore Unrestricted data so we can plot all of it, not just the data used to make our formula
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    ## Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    logPdivLWData = np.log10((DataToFit[PIndex]/DataToFit[LIndex]**WRatio)) # log (P/(L^W))
    
    # Do our first fit for determining a function for Mdot
    ## logMdot ~ logA + c log((P/L)^W) + d logM
    x0 = logPdivLWData
    x1 = logMData
    y = logMdotData
    data = pandas.DataFrame({'logPdivLW': x0, 'logM': x1, 'logMdot': y})
    # Fit the model
    model = wls("logMdot ~ logPdivLW + logM", data).fit()
    
    # Refine the previous fit using an ODR Method
    ## Define a function (four-variable linear in our case) to fit the data with.
    def fitting_func(B, x):
        return B[0]*x[0] + B[1]*x[1] + B[2] # A simple linear fit
    ## Create a model for fitting.
    linear_model = Model(fitting_func)
    # Create a RealData object using our initiated data from above.
    data = RealData((logPdivLWData, logMData), y=logMdotData) # Data: x=logL, y=logR
    ## Set up ODR with the model and data.
    odr = ODR(data, linear_model, beta0=[model._results.params[1],model._results.params[2],model._results.params[0]]) # Use the linear regression as an initial guess
    ## Run the regression.
    outPdivLW_M = odr.run()
    
    # Using the above fit, calculate the resutls for logMdot
    logMdot_PdivLM = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2]
    print( 'logMdot =', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM + ', outPdivLW_M.beta[2])
    
    # Do a fit of calculated Mdot vs observed Mdot for plotting
    x0 = logMdotData
    y = logMdot_PdivLM
    data = pandas.DataFrame({'logMdotData': x0, 'logMdot_PdivLM': y})
    # Fit the model
    model = ols("logMdot_PdivLM ~ logMdotData", data).fit()
    PdivLMLine = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    
    # Plot our initial fit, and see how it compares to observation
    
    # With that first fit done, we will start improving it by subtracting the residuals from the fit
    ResidualYAxis = logMdot_PdivLM - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    PlotTitle = specify_star_type + ' - No Errors Included' # Label the plot with whatever data set we are using
    
    # Fit logP to the residuals in a linear fit
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logPData
    y = ResidualYAxis
    data = pandas.DataFrame({'logPData': x0, 'ResidualYAxis': y})
    model = ols("ResidualYAxis ~ logPData", data).fit()
    ResidualsFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residual Fit: logMdotCalc-logMdotObsv =', ResidualsFit[0], '+', ResidualsFit[1], 'logP')
    
    # Now, we are going to start comparing the residual to various useful parameters
    ylabel = 'Mdot Residuals' # What is on the y label until we change this value
    
    # Now, remove the residuals
    NewMdotCalc = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2] - ResidualsFit[0] - ResidualsFit[1]*logPData
    ylabel = "NewlogMdotCalc"
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM - ', ResidualsFit[1], 'logP', '; W =', WRatio )

    # Simplified statement: constants + A log P + B log L + log M
    A = -ResidualsFit[0] + outPdivLW_M.beta[2]
    B = -1*outPdivLW_M.beta[0]*WRatio
    C = outPdivLW_M.beta[1]
    D = outPdivLW_M.beta[0] - ResidualsFit[1]

    #print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+ (', outPdivLW_M.beta[0] - ResidualsFit[1],') logP + (', -1*outPdivLW_M.beta[0]*WRatio,') logL + ', outPdivLW_M.beta[1], 'logM'  )
    print('NewMdotCalc =', A, '+ (', D,') logP + (', B,') logL + ', C, 'logM'  )

################# (*IMPORTANT*) Now we go through again, but this time with the mass dependence removed ##########################
    MassDependenceRemovedData = logMdotData - C*logMData # Using the above calculations, remove the mass dependence on Mdot
    DataToFit_plus_MDRD = np.vstack((DataToFit,MassDependenceRemovedData)) # Add the Mdot-ClogM results to the end of the DataToFit Array
    DataToFit_plus_MDRD = sort_star_data_MDRD(DataToFit_plus_MDRD) # Should sort data based on final element (the MDR Data)
    
    V2, W2Ratio = VWFit_Q_method (DataToFit_plus_MDRD,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound)
    
    # Restore Unrestricted data so we can plot all of it, not just the data used to make our formula
    #DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.

    # Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    logPdivLWData = logPData - W2Ratio*logLData
    
    # logMdot ~ logA + c log((P/L)^W) + d logM
    x0 = logPdivLWData
    x1 = logMData
    y = logMdotData

    data = pandas.DataFrame({'logQz': x0, 'logM': x1, 'logMdot': y})

    # Fit the model
    model = wls("logMdot ~ logQz + logM", data).fit()
    
    logMdot_PdivLM_model_results = [model._results.params[0],model._results.params[1],model._results.params[2]]
    logMdot_PdivLM = logMdot_PdivLM_model_results[1]*logPdivLWData + logMdot_PdivLM_model_results[2]*logMData + logMdot_PdivLM_model_results[0]
    print( 'logMdot =', outPdivLW_M.beta[1], 'log(P/L^W) + ', outPdivLW_M.beta[2], 'logM + ', model._results.params[0])
    
    # logMdot ~ logM + logL + logP

    x0 = logMdotData
    y = logMdot_PdivLM
    data = pandas.DataFrame({'logMdotData': x0, 'logMdot_PdivLM': y})
    model = ols("logMdot_PdivLM ~ logMdotData", data).fit()
    PdivLMLine = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    
    
     # Plot our initial fit, and see how it compares to observation
    first_fit_mdot_fig = plt.figure()
    SlopeTestNoErrors=PdivLMLine[0] + PdivLMLine[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    plot.scatter(logMdotData, logMdot_PdivLM, s=2, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        first_fit_mdot_fig.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    # With that first fit done, we will start improving it by subtracting the residuals from the fit
    ResidualYAxis = logMdot_PdivLM - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    PlotTitle = specify_star_type + ' - No Errors Included' # Label the plot with whatever data set we are using
    
    # Fit logP to the residuals in a linear fit
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logPData
    y = ResidualYAxis
    data = pandas.DataFrame({'logPData': x0, 'ResidualYAxis': y})
    model = ols("ResidualYAxis ~ logPData", data).fit()
    ResidualsFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residual Fit: logMdotCalc-logMdotObsv =', ResidualsFit[0], '+', ResidualsFit[1], 'logP')
    
    # Now, we are going to start comparing the residual to various useful parameters
    ylabel = 'Mdot Residuals' # What is on the y label until we change this value
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    first_resid_vs_P = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_P.savefig(save_directory+'Residual_Images/first_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    first_resid_vs_L = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    plot.title(PlotTitle)
    if save_figures != 0:
        first_resid_vs_L.savefig(save_directory+'Residual_Images/first_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(W2Ratio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    first_resid_vs_Q = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_Q.savefig(save_directory+'Residual_Images/first_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    first_resid_vs_M = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_M.savefig(save_directory+'Residual_Images/first_resid_vs_logM.png')
    plot.close()
    
    # Now remove the residuals again
    NewMdotCalc = logMdot_PdivLM_model_results[1]*logPdivLWData + logMdot_PdivLM_model_results[2]*logMData + logMdot_PdivLM_model_results[0] - ResidualsFit[0] - ResidualsFit[1]*logPData
    ylabel = "NewlogMdotCalc"
    print('NewMdotCalc =', -ResidualsFit[0] + logMdot_PdivLM_model_results[0], '+', logMdot_PdivLM_model_results[1], 'log(P/L^W) + ', logMdot_PdivLM_model_results[2], 'logM - ', ResidualsFit[1], 'logP', '; W =', W2Ratio )
    # Simplified statement: constants + A log P + B log L + log M
    print('NewMdotCalc =', -ResidualsFit[0] + logMdot_PdivLM_model_results[0], '+ (', logMdot_PdivLM_model_results[1] - ResidualsFit[1],') logP + (', -1*logMdot_PdivLM_model_results[1]*W2Ratio,') logL + ', logMdot_PdivLM_model_results[2], 'logM'  )
    
    # logMdot ~ logM + logL + logP
    xdataSet = logMdotData
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = xdataSet
    y = NewMdotCalc

    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})

    # Fit the model
    model = ols("NewMdotCalc ~ fittingData", data).fit()

    # Print the summary
    #print(model.summary())

    #print("\nRetrieving manually the parameter estimates:")
    #print(model._results.params)
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('NewMdotCalc Fit: NewlogMdotCalc =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logMdot_obsv')

    plot.scatter(xdataSet, NewMdotCalc, s=2, color='green')
    plot.plot(np.asarray([-9,-7,-6,-4]),np.asarray([-9,-7,-6,-4]),color='black')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.xlim(mdot_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logMdot_obsv")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        plot.savefig(save_directory+'/newMdot_vs_Mdotobsv')
    plot.close()
    
    # 
    NewResidual = NewMdotCalc - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    ylabel = 'Old Residual' # What is on the y label until we change this value

    xdataSet = NewResidual
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = xdataSet
    y = ResidualYAxis

    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})

    # Fit the model
    model = ols("NewMdotCalc ~ fittingData", data).fit()

    # Print the summary
    #print(model.summary())

    #print("\nRetrieving manually the parameter estimates:")
    #print(model._results.params)
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residuals Fit: OldResidual =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'NewResidual')

    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.plot(np.asarray([-4,-1,0,1,4]),np.asarray([-4,-1,0,1,4]),color='black')
    plot.xlim(resid_limits)
    plot.ylim(resid_limits)
    plot.xlabel("New Residual")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        plot.savefig(save_directory+'Residual_Images/first_resid_vs_second_resid.png')
    plot.close()
    
    ### RMS of Data
    RMSE = calc_rms_error (NewMdotCalc, logMdotData)
    print ('RMS Error:', RMSE)
    print ('STD Dev:', np.std(NewMdotCalc))
    
    ylabel = 'New Residual'
    ResidualYAxis = NewResidual
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData

    x0 = xdataSet
    y = ResidualYAxis

    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})

    # Fit the model
    model = ols("NewMdotCalc ~ fittingData", data).fit()

    # Print the summary

    #print("\nRetrieving manually the parameter estimates:")
    #print(model._results.params)
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')


    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        plot.savefig(save_directory+'Residual_Images/second_resid_vs_logP')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData

    x0 = xdataSet
    y = ResidualYAxis

    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})

    # Fit the model
    model = ols("NewMdotCalc ~ fittingData", data).fit()

    # Print the summary

    #print("\nRetrieving manually the parameter estimates:")
    #print(model._results.params)
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')


    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        plot.savefig(save_directory+'Residual_Images/second_resid_vs_logL')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))

    xdataSet = logQData

    x0 = xdataSet
    y = ResidualYAxis

    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})

    # Fit the model
    model = ols("NewMdotCalc ~ fittingData", data).fit()

    # Print the summary

    #print("\nRetrieving manually the parameter estimates:")
    #print(model._results.params)
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')

    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        plot.savefig(save_directory+'Residual_Images/second_resid_vs_logQz')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM

    xdataSet = logMData

    x0 = xdataSet
    y = ResidualYAxis

    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})

    # Fit the model
    model = ols("NewMdotCalc ~ fittingData", data).fit()

    # Print the summary

    #print("\nRetrieving manually the parameter estimates:")
    #print(model._results.params)
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')

    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        plot.savefig(save_directory+'Residual_Images/second_resid_vs_logM')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM

    xdataSet = logRData

    x0 = xdataSet
    y = NewMdotCalc

    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})

    # Fit the model
    model = ols("NewMdotCalc ~ fittingData", data).fit()

    # Print the summary

    #print("\nRetrieving manually the parameter estimates:")
    #print(model._results.params)
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')

    plot.scatter(logRData, NewMdotCalc, s=2, color='green')
    plot.xlim(1.5,3)
    plot.ylim(mdot_limits)
    plot.xlabel("logR")
    plot.ylabel('Mdot_calc_New')
    plot.title(PlotTitle)
    if save_figures != 0:
        plot.savefig(save_directory+'Residual_Images/second_resid_vs_logR')
    plot.close()
    
    #return 1 # Placeholder. Should change to return values of interest in an array.
    return [-ResidualsFit[0] + outPdivLW_M.beta[2], outPdivLW_M.beta[0] - ResidualsFit[1], -1*outPdivLW_M.beta[0]*WRatio, outPdivLW_M.beta[1] ], NewMdotCalc, logMdotData, [V2,W2Ratio]

In [ ]:
mdot_fit_Qz_percentile_select (M0StarData, specify_star_type ='M0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

In [ ]:
mdot_fit_Qz_percentile_select (M1StarData, specify_star_type ='M1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )
mdot_fit_Qz_percentile_select (C0StarData, specify_star_type ='C0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )
mdot_fit_Qz_percentile_select (C1StarData, specify_star_type ='C1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

In [ ]:
def mdot_fit_Qz_piecewise (data_set_pw, specify_star_type ='', number_of_ranges = 2, save_figures = 0 ):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qz_method/Sectioned/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qz_method/Sectioned/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qz_method/Sectioned/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qz_method/Sectioned/C1/'
    else:
        save_directory = 'MDot_Fits/Qz_method/Sectioned/Other/'
        
    DataToSplit = sort_star_data_Q (data_set_pw )
    #print('Debug: DataToFit:')
    data_for_sectioning = np.transpose(DataToSplit)
    #print(len(data_for_sectioning))
    # Determine split points
    split_increment = int(np.floor(len(data_for_sectioning)/number_of_ranges))
    if split_increment < 1:
        print ('ERROR: More desired ranges than Data Points')
    splitting_points = []
    split_point = split_increment # Initialize this at the lowest split point
    while split_point < number_of_ranges*split_increment:
        splitting_points.append(split_point)
        split_point = split_point + split_increment
    print('Splitting Points:', splitting_points)
    # Take data set and split into n ranges. Defaults to two.
    sectioned_data = np.transpose(np.split (data_for_sectioning, splitting_points))
    #print(sectioned_data[1])
    print('Data pts. in array:', len(sectioned_data[1]))
    returned_fit_parameters = []
    returned_Mdot_obsv_Data = []
    returned_Mdot_calc_Data = []
    returned_fit_Ws = []
    for i in range(0, number_of_ranges):
        print('*Range ',i, 'Start*')
        working_data = np.transpose(np.asarray(sectioned_data[i]))
        results = mdot_fit_Qz_percentile_select (working_data, specify_star_type, method_folder = 'Sectioned')
        returned_fit_parameters.append(results[0])
        returned_Mdot_calc_Data.append(results[1])
        returned_Mdot_obsv_Data.append(results[2])
        returned_fit_Ws.append(results[3][1])
        print('*Range',i, 'End*\n')
        
    logMdotData = np.log10(data_set_pw[MdotIndex])
    calc_MDot_Data = np.concatenate(returned_Mdot_calc_Data)
    returned_stars_obsv = 0
    for i in range(0,len(returned_Mdot_obsv_Data)):
        returned_stars_obsv = returned_stars_obsv + len(returned_Mdot_obsv_Data[i])
    returned_stars_calc = 0
    for i in range(0,len(returned_Mdot_calc_Data)):
        returned_stars_calc = returned_stars_calc + len(returned_Mdot_calc_Data[i])
    #print('DEBUG: len(data_for_sectioning)=',len(data_for_sectioning), 'returned_stars_obsv=',returned_stars_obsv, 'len(returned_stars_calc)=', returned_stars_calc)
    
    mdot_calc_vs_obsv = plt.figure()
    #print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    legend_labels = ['1:1 Line']
    for i in range (0, number_of_ranges):
        plot.scatter(returned_Mdot_obsv_Data[i], returned_Mdot_calc_Data[i], s=2)
        #legend_labels.append('Range ' + str(i) + '(' + str(np.min(returned_Mdot_obsv_Data[i]))[:5] + ' -- ' + str(np.max(returned_Mdot_obsv_Data[i]))[:5]+')')
        legend_labels.append(str(np.mean(returned_Mdot_obsv_Data[i]))[:5])
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.legend(legend_labels)
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        mdot_calc_vs_obsv.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    print('W Values:', returned_fit_Ws)
    

In [ ]:
mdot_fit_Qz_piecewise (M0StarData, specify_star_type ='M0', number_of_ranges = 10, save_figures = 1 )

In [ ]:
mdot_fit_Qz_piecewise (M1StarData, specify_star_type ='M1', number_of_ranges = 10, save_figures = 1 )

In [ ]:
mdot_fit_Qz_piecewise (C0StarData, specify_star_type ='C0', number_of_ranges = 10, save_figures = 1 )

In [ ]:
mdot_fit_Qz_piecewise (C1StarData, specify_star_type ='C1', number_of_ranges = 10, save_figures = 1 )

In [ ]:
# Functions to sort a list
def examine_index_Qq(elem): 
    # Sorting Key. Will sort a list based on the specified index hard coded here.
    return elem[MIndex]

def sort_star_data_Qq(DataToSort):
    TransposedArray=np.transpose(DataToSort)
    TempList=TransposedArray.tolist()
    TempList=sorted(TempList,key = operator.itemgetter(MIndex))
    return np.transpose(np.asarray(TempList))

def MassRangeSelect_Qq (DataToTrim, LowerBound, UpperBound ):
    return np.transpose(np.transpose(sort_star_data_Qq(DataToTrim))[LowerBound-1:UpperBound-1])

In [ ]:
def VWFit_Qq_method (data_set, specify_star_type ='', VW_fit_lower_percentile_bound=0, VW_fit_upper_percentile_bound = 100 , save_figures_VW = 0, method_folder = 'Percentile_Range'): 
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/C1/'
    else:
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/Other/'
    
    # First, set up data for the initial VW Fit
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    PercentileLower= int(np.floor(VW_fit_lower_percentile_bound/100*len(np.transpose(DataToFit))))
    PercentileUpper = int(np.ceil(VW_fit_upper_percentile_bound/100*len(np.transpose(DataToFit))))
    #print(VW_fit_lower_percentile_bound, 'th Percentile:', PercentileLower,VW_fit_upper_percentile_bound, 'th Percentile:', PercentileUpper)
    
    if PercentileLower !=0 and PercentileUpper != 100:
        DataToFit = MassRangeSelect_Q (DataToFit, PercentileLower, PercentileUpper )
    
    #print('DEBUG: DataToFit:', DataToFit)
    
    
    # Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    
    # Next, do the fit to obtain V and W
    ## logP ~ V + W log L
    ## First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logLData
    y = logPData
    data = pandas.DataFrame({'logL': x0, 'logP': y})
    ## Fit the model
    model = wls("logP ~ logL", data).fit()
    ## Save the model results into variables
    V = model._results.params[0]
    WRatio = model._results.params[1]
    print('W=',WRatio)
    
    # Visualize the fit
    VWFit_fig = plt.figure()
    logPdivLWData = np.log10((DataToFit[PIndex]/DataToFit[LIndex]**WRatio)) # log (P/(L^W))
    print('logP=', V, '+', WRatio, 'logL')
    plot.scatter(logLData,logPData,s=2,color='blue')
    coordinate_list = np.asarray([3,3.5,4])
    logLvlogP_predict=V + WRatio *coordinate_list
    plot.plot(coordinate_list,logLvlogP_predict, color='black')
    plot.xlabel("logL")
    plot.ylabel("logP")
    plot.xlim(coordinate_list[0], coordinate_list[-1])
    plot.ylim(1.7, 3)
    plot.title(specify_star_type+ ' - logP = V + W logL')
    if save_figures_VW != 0:
        VWFit_fig.savefig(save_directory+'VWFit.png')
    plot.close()
    
    return V, WRatio



In [ ]:
# This function takes a set of stars (already sorted by type and pulsation mode) and fits a mass loss fit using the stars in a given percentile range as the basis for the fit
def mdot_fit_Qq_percentile_select (data_set, specify_star_type ='', VW_fit_lower_percentile_bound=0, VW_fit_upper_percentile_bound = 100, save_figures = 0, method_folder = 'Percentile_Range'):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/C1/'
    else:
        save_directory = 'MDot_Fits/Qq_method/'+method_folder+'/Other/'
    # Create directories if they do not exist
    
    # Use an algorithm to determine V and W. Defined earlier in the code.
    if method_folder == 'Percentile_Range':
        V, WRatio = VWFit_Qq_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures)
    else:
        V, WRatio = VWFit_Qq_method (data_set,specify_star_type,VW_fit_lower_percentile_bound, VW_fit_upper_percentile_bound, save_figures_VW = save_figures, method_folder = method_folder)
    # Restore Unrestricted data so we can plot all of it, not just the data used to make our formula
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    ## Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    logPdivLWData = np.log10((DataToFit[PIndex]/DataToFit[LIndex]**WRatio)) # log (P/(L^W))
    
    # Do our first fit for determining a function for Mdot
    ## logMdot ~ logA + c log((P/L)^W) + d logM
    x0 = logPdivLWData
    x1 = logMData
    y = logMdotData
    data = pandas.DataFrame({'logPdivLW': x0, 'logM': x1, 'logMdot': y})
    # Fit the model
    model = wls("logMdot ~ logPdivLW + logM", data).fit()
    
    # Refine the previous fit using an ODR Method
    ## Define a function (four-variable linear in our case) to fit the data with.
    def fitting_func(B, x):
        return B[0]*x[0] + B[1]*x[1] + B[2] # A simple linear fit
    ## Create a model for fitting.
    linear_model = Model(fitting_func)
    # Create a RealData object using our initiated data from above.
    data = RealData((logPdivLWData, logMData), y=logMdotData) # Data: x=logL, y=logR
    ## Set up ODR with the model and data.
    odr = ODR(data, linear_model, beta0=[model._results.params[1],model._results.params[2],model._results.params[0]]) # Use the linear regression as an initial guess
    ## Run the regression.
    outPdivLW_M = odr.run()
    
    # Using the above fit, calculate the resutls for logMdot
    logMdot_PdivLM = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2]
    print( 'logMdot =', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM + ', outPdivLW_M.beta[2])
    
    # Do a fit of calculated Mdot vs observed Mdot for plotting
    x0 = logMdotData
    y = logMdot_PdivLM
    data = pandas.DataFrame({'logMdotData': x0, 'logMdot_PdivLM': y})
    # Fit the model
    model = ols("logMdot_PdivLM ~ logMdotData", data).fit()
    PdivLMLine = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    
    # Plot our initial fit, and see how it compares to observation
    first_fit_mdot_fig = plt.figure()
    SlopeTestNoErrors=PdivLMLine[0] + PdivLMLine[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    plot.scatter(logMdotData, logMdot_PdivLM, s=2, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        first_fit_mdot_fig.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    # With that first fit done, we will start improving it by subtracting the residuals from the fit
    ResidualYAxis = logMdot_PdivLM - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    PlotTitle = specify_star_type + ' - No Errors Included' # Label the plot with whatever data set we are using
    
    # Fit logP to the residuals in a linear fit
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logPData
    y = ResidualYAxis
    data = pandas.DataFrame({'logPData': x0, 'ResidualYAxis': y})
    model = ols("ResidualYAxis ~ logPData", data).fit()
    ResidualsFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residual Fit: logMdotCalc-logMdotObsv =', ResidualsFit[0], '+', ResidualsFit[1], 'logP')
    
    # Now, we are going to start comparing the residual to various useful parameters
    ylabel = 'Mdot Residuals' # What is on the y label until we change this value
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    first_resid_vs_P = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_P.savefig(save_directory+'Residual_Images/first_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    first_resid_vs_L = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    plot.title(PlotTitle)
    if save_figures != 0:
        first_resid_vs_L.savefig(save_directory+'Residual_Images/first_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    first_resid_vs_Q = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_Q.savefig(save_directory+'Residual_Images/first_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    first_resid_vs_M = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_M.savefig(save_directory+'Residual_Images/first_resid_vs_logM.png')
    plot.close()
    
    # Now, remove the residuals
    NewMdotCalc = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2] - ResidualsFit[0] - ResidualsFit[1]*logPData
    ylabel = "NewlogMdotCalc"
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM - ', ResidualsFit[1], 'logP', '; W =', WRatio )
    # Simplified statement: constants + A log P + B log L + log M
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+ (', outPdivLW_M.beta[0] - ResidualsFit[1],') logP + (', -1*outPdivLW_M.beta[0]*WRatio,') logL + ', outPdivLW_M.beta[1], 'logM'  )
    
    # logMdot ~ logM + logL + logP
    xdataSet = logMdotData
    x0 = xdataSet
    y = NewMdotCalc
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('NewMdotCalc Fit: NewlogMdotCalc =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logMdot_obsv')
    newmdotcalc_vs_mdotobsv = plt.figure()
    plot.scatter(xdataSet, NewMdotCalc, s=2, color='green')
    plot.plot(np.asarray([-9,-7,-6,-4]),np.asarray([-9,-7,-6,-4]),color='black')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.xlim(mdot_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logMdot_obsv")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        newmdotcalc_vs_mdotobsv.savefig(save_directory+'newmdotcalc_vs_mdotobsv.png')
    plot.close()
    
    # Now, we're going to compare residuals
    
    NewResidual = NewMdotCalc - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    ylabel = 'Old Residual' # What is on the y label until we change this value
    xdataSet = NewResidual
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residuals Fit: OldResidual =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'NewResidual')
    second_resid_vs_first_resid = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-4,-1,0,1,4]),np.asarray([-4,-1,0,1,4]),color='black')
    plot.xlim(resid_limits)
    plot.ylim(resid_limits)
    plot.xlabel("New Residual")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_first_resid.savefig(save_directory+'Residual_Images/second_resid_vs_first_resid.png')
    plot.close()
    
    ### RMS of Data
    RMSE = calc_rms_error (NewMdotCalc, logMdotData)
    print ('RMS Error:', RMSE)
    print ('STD Dev:', np.std(NewMdotCalc))
    
    ylabel = 'Second Residual'
    ResidualYAxis = NewResidual
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    second_resid_vs_logP = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logP.savefig(save_directory+'Residual_Images/second_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    second_resid_vs_logL = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logL.savefig(save_directory+'Residual_Images/second_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    second_resid_vs_logQ = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logQ.savefig(save_directory+'Residual_Images/second_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    second_resid_vs_logM = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logM.savefig(save_directory+'Residual_Images/second_resid_vs_logM.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logR
    xdataSet = logRData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logR')
    second_resid_vs_logR = plt.figure()
    plot.scatter(logRData, NewMdotCalc, s=2, color='green')
    plot.xlim(radius_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logR")
    plot.ylabel('Mdot_calc_New')
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logR.savefig(save_directory+'Residual_Images/second_resid_vs_logR.png')
    plot.close()
    
    #return 1 # Placeholder. Should change to return values of interest in an array.
    return [-ResidualsFit[0] + outPdivLW_M.beta[2], outPdivLW_M.beta[0] - ResidualsFit[1], -1*outPdivLW_M.beta[0]*WRatio, outPdivLW_M.beta[1] ], NewMdotCalc, logMdotData, [V,WRatio]
    # What data do we need to carry over for a piecewise?: RawMdotData, NewMdotcalc coefficients, NewMdotCalc of input data

In [ ]:
mdot_fit_Qq_percentile_select (M0StarData, specify_star_type ='M0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

In [ ]:
mdot_fit_Qq_percentile_select (M1StarData, specify_star_type ='M1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

In [ ]:
mdot_fit_Qq_percentile_select (C0StarData, specify_star_type ='C0', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

In [ ]:
mdot_fit_Qq_percentile_select (C1StarData, specify_star_type ='C1', VW_fit_lower_percentile_bound=40, VW_fit_upper_percentile_bound = 60 )

In [ ]:
def mdot_fit_Qq_piecewise (data_set_pw, specify_star_type ='', number_of_ranges = 2, save_figures = 0 ):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Qq_method/Sectioned/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Qq_method/Sectioned/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Qq_method/Sectioned/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Qq_method/Sectioned/C1/'
    else:
        save_directory = 'MDot_Fits/Qq_method/Sectioned/Other/'
    
    
    DataToSplit = sort_star_data_Qq (data_set_pw )
    #print('Debug: DataToFit:')
    data_for_sectioning = np.transpose(DataToSplit)
    #print(len(data_for_sectioning))
    # Determine split points
    split_increment = int(np.floor(len(data_for_sectioning)/number_of_ranges))
    if split_increment < 1:
        print ('ERROR: More desired ranges than Data Points')
    splitting_points = []
    split_point = split_increment # Initialize this at the lowest split point
    while split_point < number_of_ranges*split_increment:
        splitting_points.append(split_point)
        split_point = split_point + split_increment
    print('Splitting Points:', splitting_points)
    # Take data set and split into n ranges. Defaults to two.
    sectioned_data = np.transpose(np.split (data_for_sectioning, splitting_points))
    partition_check (DataToSplit,sectioned_data)
    
    #print(sectioned_data[1])
    print('Data pts. in array:', len(sectioned_data[1]))
    returned_fit_parameters = []
    returned_Mdot_obsv_Data = []
    returned_Mdot_calc_Data = []
    returned_fit_Ws = []
    for i in range(0, number_of_ranges):
        print('*Range ',i, 'Start*')
        working_data = np.transpose(np.asarray(sectioned_data[i]))
        results = mdot_fit_Qq_percentile_select (working_data, specify_star_type, method_folder = 'Sectioned')
        returned_fit_parameters.append(results[0])
        returned_Mdot_calc_Data.append(results[1])
        returned_Mdot_obsv_Data.append(results[2])
        returned_fit_Ws.append(results[3][1])
        print('*Range',i, 'End*\n')
        
    logMdotData = np.log10(data_set_pw[MdotIndex])
    calc_MDot_Data = np.concatenate(returned_Mdot_calc_Data)
    returned_stars_obsv = 0
    for i in range(0,len(returned_Mdot_obsv_Data)):
        returned_stars_obsv = returned_stars_obsv + len(returned_Mdot_obsv_Data[i])
    returned_stars_calc = 0
    for i in range(0,len(returned_Mdot_calc_Data)):
        returned_stars_calc = returned_stars_calc + len(returned_Mdot_calc_Data[i])
    #print('DEBUG: len(data_for_sectioning)=',len(data_for_sectioning), 'returned_stars_obsv=',returned_stars_obsv, 'len(returned_stars_calc)=', returned_stars_calc)
    mdot_calc_vs_obsv = plt.figure()
    #print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    legend_labels = ['1:1 Line']
    for i in range (0, number_of_ranges):
        plot.scatter(returned_Mdot_obsv_Data[i], returned_Mdot_calc_Data[i], s=2)
        legend_labels.append(str(np.min(returned_Mdot_obsv_Data[i]))[:5] + ' -- ' + str(np.max(returned_Mdot_obsv_Data[i]))[:5])
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.legend(legend_labels)
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        mdot_calc_vs_obsv.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    print('W Values:', returned_fit_Ws)

In [ ]:
mdot_fit_Qq_piecewise (M0StarData, specify_star_type ='M0', number_of_ranges = 5, save_figures = 1 )

In [ ]:
mdot_fit_Qq_piecewise (M1StarData, specify_star_type ='M1', number_of_ranges = 5, save_figures = 1 )

In [ ]:
mdot_fit_Qq_piecewise (C0StarData, specify_star_type ='C0', number_of_ranges = 5, save_figures = 1 )

In [ ]:
mdot_fit_Qq_piecewise (C1StarData, specify_star_type ='C1', number_of_ranges = 5, save_figures = 1 )